# Merge cleaned data and review text

Run `data_cleaning.ipynb` and `extract_html_data.ipynb` first, then run these
cells in order. This notebook performs the work of `scripts/merge_reviews.py`
with the same matching and exclusion rules. All processing code is included
here and uses only Python's standard library.

Match on normalized artist, album, and publication date. Accept a review only
when there is one candidate and its author and numeric score agree. Save
`outputs/full_review.csv`, unresolved rows in `outputs/missed_on_merge/`, and
discarded rows in `outputs/discarded_on_merge/`. Each accepted `source_id` is
used once. Input CSVs are preserved.

## 1. Imports and paths

Extraction support files are read from `outputs/extracting_text/`. Edit the
paths below if you want to use different input CSVs or an output directory.

In [1]:
from collections import Counter, defaultdict
import csv
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path
from pprint import pprint
import unicodedata


In [2]:
# Locate the repository from its root, notebooks/, or a deeper directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data.csv").is_file() and (path / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook with this repository as the working directory.")

OUTPUTS_DIR = REPO_ROOT / "outputs"
EXTRACTION_DIR = OUTPUTS_DIR / "extracting_text"

DATA_CLEAN_CSV = OUTPUTS_DIR / "data_clean.csv"
REVIEW_TEXT_CSV = OUTPUTS_DIR / "review_text.csv"
DISCARDED_HTML_CSV = EXTRACTION_DIR / "discarded_html.csv"

print("Clean metadata:", DATA_CLEAN_CSV)
print("Review text:", REVIEW_TEXT_CSV)
print("Discarded HTML report:", DISCARDED_HTML_CSV)
print("Output directory:", OUTPUTS_DIR)


Clean metadata: /home/ai4covid/test_stuff_2/outputs/data_clean.csv
Review text: /home/ai4covid/test_stuff_2/outputs/review_text.csv
Discarded HTML report: /home/ai4covid/test_stuff_2/outputs/extracting_text/discarded_html.csv
Output directory: /home/ai4covid/test_stuff_2/outputs


## 2. Schemas, normalization, and consistency checks

Normalize Unicode, curly quotes, whitespace, and case for matching while
retaining the original column values in the output. Do not remove punctuation
or guess replacements for spreadsheet-converted album names.

Numeric scores such as `8` and `8.0` are equivalent. Missing or conflicting
authors and invalid or conflicting scores prevent a match.

In [3]:
IDENTITY_FIELDS = [
    "source_id", "artist", "album", "reviewdate", "reviewauthor", "score"
]


REVIEW_FIELDS = IDENTITY_FIELDS + ["description", "review"]


MATCH_FIELDS = ["artist", "album", "reviewdate", "reviewauthor", "score"]


REQUESTED_EXCLUSIONS = {("prince", "1999", "2019-12-02")}


In [4]:
QUOTE_MAP = str.maketrans({
    "‘": "'", "’": "'", "‛": "'", "′": "'", "“": '"', "”": '"', "″": '"',
})


def normalize_name(value):
    """Normalize Unicode, quotes, whitespace, and case without losing punctuation."""
    return " ".join(
        unicodedata.normalize("NFKC", value or "").translate(QUOTE_MAP).split()
    ).casefold()


def review_date(value):
    return date.fromisoformat((value or "").strip()[:10]).isoformat()


def review_score(value):
    try:
        score = Decimal((value or "").strip())
    except InvalidOperation as exc:
        raise ValueError(f"invalid review score: {value!r}") from exc
    if not score.is_finite() or not 0 <= score <= 10:
        raise ValueError(f"invalid review score: {value!r}")
    return score


def match_key(row):
    artist, album = (normalize_name(row.get(field)) for field in ("artist", "album"))
    if not artist or not album:
        raise ValueError("missing artist or album")
    return artist, album, review_date(row.get("reviewdate"))


def duplicate_keys(rows):
    counts = Counter()
    for row in rows:
        try:
            counts[match_key(row)] += 1
        except ValueError:
            continue
    return {key for key, count in counts.items() if count > 1}


In [5]:
def read_csv(path, required=()):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames or []
        missing = set(required) - set(fields)
        if missing:
            raise ValueError(f"{path}: missing columns {sorted(missing)}")
        if len(set(fields)) != len(fields):
            raise ValueError(f"{path}: duplicate column names")
        return fields, list(reader)


def write_csv(path, fields, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


In [6]:
def consistency_issues(data_row, text_row):
    issues = []
    data_author = normalize_name(data_row.get("reviewauthor"))
    text_author = normalize_name(text_row.get("reviewauthor"))
    if not data_author or not text_author:
        issues.append("missing_author")
    elif data_author != text_author:
        issues.append("author_mismatch")
    try:
        if review_score(data_row.get("score")) != review_score(text_row.get("score")):
            issues.append("score_mismatch")
    except ValueError:
        issues.append("invalid_score")
    return ";".join(issues)


## 3. Load inputs and identify exclusion groups

Discard **every** metadata row in a repeated normalized artist/album/date
group, along with corresponding review-text rows. Also exclude the duplicate
HTML cases listed by extraction. Prince's December 2, 2019 review of `1999` is
explicitly excluded; the April 30, 2016 review remains eligible.

In [7]:
data_path = DATA_CLEAN_CSV
text_path = REVIEW_TEXT_CSV
output_dir = OUTPUTS_DIR
discarded_html_path = DISCARDED_HTML_CSV

data_fields, data_rows = read_csv(data_path, MATCH_FIELDS)

text_fields, text_rows = read_csv(text_path, REVIEW_FIELDS)

if {"source_id", "description", "review"} & set(data_fields):
    raise ValueError("cleaned metadata already contains output-only review columns")

repeated_data_keys = duplicate_keys(data_rows)

discarded_html_keys = set()

if discarded_html_path is not None:
    _, html_rows = read_csv(discarded_html_path, MATCH_FIELDS)
    discarded_html_keys = {match_key(row) for row in html_rows}

print(f"Loaded {len(data_rows):,} metadata rows and {len(text_rows):,} review-text rows.")
print(f"Found {len(repeated_data_keys):,} repeated metadata key/date groups.")


Loaded 16,811 metadata rows and 21,271 review-text rows.
Found 6 repeated metadata key/date groups.


## 4. Separate eligible, discarded, and invalid rows

Keep reasons for all rejected rows. Invalid match fields are unresolved.
Repeated source IDs and the requested exclusion groups are discarded.

In [8]:
data_discarded, text_discarded = [], []

data_unmatched, text_unmatched = [], []

eligible_data, eligible_text = [], []

id_counts = Counter(row["source_id"] for row in text_rows)

for rows, eligible, discarded, unmatched, is_text in (
    (data_rows, eligible_data, data_discarded, data_unmatched, False),
    (text_rows, eligible_text, text_discarded, text_unmatched, True),
):
    for row in rows:
        try:
            key = match_key(row)
        except ValueError:
            unmatched.append({**row, "merge_reason": "invalid_match_fields"})
            continue
        reason = None
        if key in repeated_data_keys:
            reason = "duplicate_metadata_key"
        elif key in REQUESTED_EXCLUSIONS:
            reason = "requested_exclusion_no_source_review"
        elif key in discarded_html_keys:
            reason = "duplicate_html_review"
        elif is_text and id_counts[row["source_id"]] > 1:
            reason = "duplicate_source_id"
        if reason:
            discarded.append({**row, "discard_reason": reason})
        else:
            eligible.append((key, row))

print(f"Eligible metadata rows: {len(eligible_data):,}; eligible text rows: {len(eligible_text):,}")
print(f"Discarded metadata rows: {len(data_discarded):,}; discarded text rows: {len(text_discarded):,}")


Eligible metadata rows: 16,797; eligible text rows: 21,265
Discarded metadata rows: 14; discarded text rows: 6


## 5. Find unique matches and validate them

Index text rows by normalized artist, album, and date. Multiple candidates,
date mismatches, author mismatches, and score mismatches stay unresolved; no
fallback joins on artist and album alone are used.

In [9]:
text_by_key = defaultdict(list)

for key, row in eligible_text:
    text_by_key[key].append(row)

text_pairs = {key[:2] for key, _ in eligible_text}

data_pairs = {key[:2] for key, _ in eligible_data}

merged = []

matched_ids = set()

failed_keys = {}


In [10]:
for key, data_row in eligible_data:
    candidates = text_by_key.get(key, [])
    if not candidates:
        reason = "review_date_mismatch" if key[:2] in text_pairs else "no_matching_review"
    elif len(candidates) != 1:
        reason = "ambiguous_review_match"
    else:
        text_row = candidates[0]
        reason = consistency_issues(data_row, text_row)
        if not reason:
            source_id = text_row["source_id"]
            if source_id in matched_ids:
                raise ValueError(f"source ID would be linked more than once: {source_id}")
            matched_ids.add(source_id)
            merged.append({
                **data_row, "source_id": source_id,
                "description": text_row["description"], "review": text_row["review"],
            })
            continue
    failed_keys[key] = reason
    data_unmatched.append({**data_row, "merge_reason": reason})

for key, text_row in eligible_text:
    if text_row["source_id"] in matched_ids:
        continue
    reason = failed_keys.get(key)
    if not reason:
        reason = "review_date_mismatch" if key[:2] in data_pairs else "no_matching_metadata"
    text_unmatched.append({**text_row, "merge_reason": reason})

assert len({row["source_id"] for row in merged}) == len(merged)
print(f"Accepted {len(merged):,} unique review links.")


Accepted 16,222 unique review links.


## 6. Save the merged CSV and all reports

`full_review.csv` retains the cleaned metadata columns and adds `source_id`,
`description`, and `review`. Both unresolved CSVs retain their input columns
and add `merge_reason`. Both discarded CSVs add `discard_reason`.

In [11]:
write_csv(output_dir / "full_review.csv", data_fields + ["source_id", "description", "review"], merged)

for filename, fields, rows in (
    ("data_clean_unmatched.csv", data_fields, data_unmatched),
    ("review_text_unmatched.csv", text_fields, text_unmatched),
):
    write_csv(output_dir / "missed_on_merge" / filename, fields + ["merge_reason"], rows)

for filename, fields, rows in (
    ("data_clean_discarded.csv", data_fields, data_discarded),
    ("review_text_discarded.csv", text_fields, text_discarded),
):
    write_csv(output_dir / "discarded_on_merge" / filename, fields + ["discard_reason"], rows)

assert len(data_rows) == len(merged) + len(data_unmatched) + len(data_discarded)

assert len(text_rows) == len(merged) + len(text_unmatched) + len(text_discarded)


In [12]:
stats = {
        "merged": len(merged),
        "data_unmatched": len(data_unmatched), "text_unmatched": len(text_unmatched),
        "data_discarded": len(data_discarded), "text_discarded": len(text_discarded),
        "duplicate_metadata_groups": len(repeated_data_keys),
        "data_unmatched_reasons": dict(Counter(row["merge_reason"] for row in data_unmatched)),
        "data_discard_reasons": dict(Counter(row["discard_reason"] for row in data_discarded)),
    }

pprint(stats)
print("Merged CSV:", OUTPUTS_DIR / "full_review.csv")
print("Unresolved rows:", OUTPUTS_DIR / "missed_on_merge")
print("Discarded rows:", OUTPUTS_DIR / "discarded_on_merge")


{'data_discard_reasons': {'duplicate_html_review': 2,
                          'duplicate_metadata_key': 12},
 'data_discarded': 14,
 'data_unmatched': 575,
 'data_unmatched_reasons': {'no_matching_review': 575},
 'duplicate_metadata_groups': 6,
 'merged': 16222,
 'text_discarded': 6,
 'text_unmatched': 5043}
Merged CSV: /home/ai4covid/test_stuff_2/outputs/full_review.csv
Unresolved rows: /home/ai4covid/test_stuff_2/outputs/missed_on_merge
Discarded rows: /home/ai4covid/test_stuff_2/outputs/discarded_on_merge


## 7. Inspect a few accepted links

The original metadata values are retained. `source_id` links each accepted
row to `review_text.csv`, the extraction manifest, and its text file.

In [13]:
for row in merged[:5]:
    pprint({field: row[field] for field in [
        "source_id", "artist", "album", "reviewdate", "reviewauthor", "score"
    ]})


{'album': 'Studio 1',
 'artist': 'Studio 1',
 'reviewauthor': 'Andy Battaglia',
 'reviewdate': '2009-02-18',
 'score': '8.5',
 'source_id': '37ab31b68172e82813f2a369f582cbaa'}
{'album': 'The Great Santa Barbara Oil Slick',
 'artist': 'John Fahey',
 'reviewauthor': 'Mark Richardson',
 'reviewdate': '2005-02-13',
 'score': '8.2',
 'source_id': '262b7d646d70d63ae9dda3963574311e'}
{'album': 'Too Much Guitar',
 'artist': 'Reigning Sound',
 'reviewauthor': 'Stephen M. Deusner',
 'reviewdate': '2004-08-19',
 'score': '8.3',
 'source_id': '63d7991eedbf90992723439b6b87ff0b'}
{'album': 'After the Last',
 'artist': 'The Red Thread',
 'reviewauthor': 'Chris Dahlen',
 'reviewdate': '2003-07-17',
 'score': '7.3',
 'source_id': 'ece0303b76fc865d6d8429b558fd8680'}
{'album': 'With Blasphemy So Heartfelt',
 'artist': 'Jessica Lea Mayfield',
 'reviewauthor': 'Stephen M. Deusner',
 'reviewdate': '2008-11-11',
 'score': '8.2',
 'source_id': 'e7a2b5ec09d43c300f2ba9381e1b06fe'}
